# Phase 1 LoRA CPT — Qwen3.5-4B-Base (Colab)

Continued pretrain with **Unsloth BF16 LoRA** on `Aniket200325/coder-pretrain-60gb`.

**Hardware:** prefer **A100 40GB** (5–7 compute units/hr), not 80GB.
**Sessions:** Colab ~**12h** limit → resume from Hub/Drive `LATEST` each time.

### Smoke (~10 min)
Uses `--token_budget 1000000 --max_steps 50`.

### Full run
Target **~5B tokens**. After the first hour, check `FIRST_HOUR_PROJECTION` in the logs; if far below 5B, restart with `--max_seq_len 2048`.

See `FINE_TUNE_DECISIONS.md` and `train_phase1.py`.

## 1. Install (transformers v5 + Unsloth; no QLoRA)

In [ ]:
# Unsloth Colab install (adjust if Unsloth docs change)
!pip install -q --upgrade pip
!pip install -q "torch" "transformers>=5.0.0" "datasets>=3.0.0" "trl" "peft" "accelerate" "huggingface_hub" "safetensors"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

import transformers, torch
print("transformers", transformers.__version__, "torch", torch.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, "Need transformers v5 for Qwen3.5"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Auth + paths (Hub token + optional Drive for resume)

In [ ]:
import os
from pathlib import Path

# --- fill these ---
HF_TOKEN = ""  # write-scoped token, or leave empty and use Colab Secrets / userdata
HUB_MODEL_ID = "YOUR_USER/coder-qwen35-4b-phase1-lora"  # private adapter repo
USE_DRIVE = True
REPO_DIR = Path("/content/Coder")  # clone or upload the fine-tune/ folder here

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")

assert HF_TOKEN, "Set HF_TOKEN (cell or Colab Secrets)"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

DRIVE_CKPT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CKPT = "/content/drive/MyDrive/coder-phase1-lora"
    Path(DRIVE_CKPT).mkdir(parents=True, exist_ok=True)
    print("Drive ckpt:", DRIVE_CKPT)

OUT_DIR = Path("/content/ckpts/phase1-lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Local out:", OUT_DIR)

## 3. Fetch training code

Either clone your GitHub repo or upload the `fine-tune/` folder so `train_phase1.py` is importable.

In [ ]:
# Option A: clone (edit URL/branch)
# !git clone https://github.com/Aniket25042003/Coder.git {REPO_DIR}

# Option B: if you already uploaded fine-tune/ to /content/fine-tune
import sys
from pathlib import Path

candidates = [
    Path("/content/Coder/fine-tune"),
    Path("/content/fine-tune"),
    Path("/content/Coder"),
]
CODE_DIR = None
for c in candidates:
    if (c / "train_phase1.py").exists():
        CODE_DIR = c
        break
assert CODE_DIR is not None, "Place train_phase1.py under /content/fine-tune or /content/Coder/fine-tune"
sys.path.insert(0, str(CODE_DIR))
print("Using", CODE_DIR)

## 4a. Smoke test (~10 min)

Verify Unsloth load, packing, checkpoint write, and resume before a long session.

In [ ]:
import subprocess, shlex

smoke_cmd = f"""
python {CODE_DIR}/train_phase1.py \
  --model unsloth/Qwen3.5-4B-Base \
  --dataset Aniket200325/coder-pretrain-60gb \
  --max_seq_len 2048 \
  --token_budget 1000000 \
  --max_steps 50 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 1 \
  --save_steps 25 \
  --logging_steps 5 \
  --ckpt_minutes 5 \
  --output_dir {OUT_DIR}-smoke \
  --resume none \
  --no_push_to_hub
""".strip()
print(smoke_cmd)
subprocess.check_call(shlex.split(smoke_cmd))

## 4b. Full Phase 1 session (resume-aware)

- Re-run this cell on every new Colab session (`--resume auto`).
- Prefer **A100 40GB**.
- Before the ~12h kill (~11h mark), ensure a save happened (`ckpt_minutes=30` + `save_steps`).
- Watch **FIRST_HOUR_PROJECTION** after ~60 minutes.

In [ ]:
import subprocess, shlex, time

SESSION_START = time.time()
MAX_SESSION_SEC = 11 * 3600  # push final save before typical 12h cutoff

drive_arg = f"--drive_ckpt_dir {DRIVE_CKPT}" if DRIVE_CKPT else ""
hub_arg = f"--hub_model_id {HUB_MODEL_ID}" if HUB_MODEL_ID and "YOUR_USER" not in HUB_MODEL_ID else "--no_push_to_hub"

full_cmd = f"""
python {CODE_DIR}/train_phase1.py \
  --model unsloth/Qwen3.5-4B-Base \
  --dataset Aniket200325/coder-pretrain-60gb \
  --max_seq_len 4096 \
  --token_budget 5000000000 \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 4 \
  --learning_rate 1e-4 \
  --save_steps 250 \
  --ckpt_minutes 30 \
  --remaining_colab_hours 45 \
  --output_dir {OUT_DIR} \
  --resume auto \
  {drive_arg} \
  {hub_arg}
""".strip()
print(full_cmd)
print(f"Session soft limit: {MAX_SESSION_SEC/3600:.0f}h from now; rely on timed ckpts + resume next session.")
subprocess.check_call(shlex.split(full_cmd))

## 5. After disconnect / next session

1. Runtime → reconnect, pick **A100 40GB** again.
2. Re-run install + auth + code-dir cells.
3. Re-run **4b** only (`--resume auto` picks Drive/local `LATEST`).
4. Stop when logs show token budget reached or credits are low.

If first-hour projection is weak: use `--max_seq_len 2048` and raise `--per_device_train_batch_size`.